# Emperor Smart Solutions (ESS) — Fine-Tuning Pipeline (QLoRA)
This notebook trains a business-only, sales-optimized AI model using 4-bit QLoRA on `meta-llama/Llama-3.2-1B-Instruct` or `meta-llama/Llama-3.2-3B-Instruct`.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch transformers peft trl datasets bitsandbytes accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

BASE_MODEL = 'meta-llama/Llama-3.2-1B-Instruct'
OUTPUT_DIR = './ess_llama3.2_lora'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
dataset = load_dataset('json', data_files={
    'train': 'finetuning_train.jsonl',
    'validation': 'finetuning_val.jsonl'
})

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    num_train_epochs=3,
    logging_steps=5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    fp16=False,
    bf16=True,
    optim='paged_adamw_8bit',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=peft_config,
    dataset_text_field='messages',
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Training complete! Artifacts saved to', OUTPUT_DIR)